# SC-23-Cross-Chain-Rust - Interoperabilite Cross-Chain

[<< Solana & Anchor](../05-Alternative-Chains/SC-22-Solana-Anchor-Python.ipynb) | [Testnet Deploy >>](SC-24-Testnet-Deploy-Python.ipynb) | [jumeau Python](SC-23-Cross-Chain-Python.ipynb)

***

## Objectifs d'apprentissage

1. Comprendre les enjeux de l'**interoperabilite**
2. Utiliser **Chainlink CCIP** pour les cross-chain messages
3. Implementer un **bridge** simple
4. Comprendre les **securite** cross-chain

### Prerequis

- [SC-3](../01-Solidity-Foundation/SC-03-Solidity-Basics-Python.ipynb) a [SC-10](../02-Solidity-Advanced/SC-10-Account-Abstraction-Python.ipynb) completes
- Notions de bridges et messaging cross-chain
- Compte sur un testnet Ethereum (Sepolia recommande)

### Duree estimee : 45 minutes

***

## 1. Enjeux Cross-Chain

L'interoperabilite entre blockchains est un defi majeur.

### Concepts cross-chain

| Probleme | Description |
|---|---|
| Finalite | Verifier la finalite sur la source |
| Securite | Empecher les double-spend |
| Latence | Temps de confirmation |
| Cout | Gas sur les deux chains |
| Oracle | Obtenir l'etat d'une autre chain |

**Solutions :**
- **Lock & Mint** : Lock tokens A, mint wrapped A' sur B
- **Burn & Mint** : Burn wrapped A', unlock A sur source
- **Liquidity Pools** : Swap via pools sur chaque chain
- **Light Clients** : Verifier les headers distamment

**Protocoles populaires :**
- **Chainlink CCIP** : Oracle-based messaging
- **LayerZero** : Omnichain interoperability
- **Axelar** : Cross-chain gateway
- **Wormhole** : Guardian network
- **Hyperlane** : Permissionless interoperability

***

## 2. Chainlink CCIP

In [2]:
// Chainlink CCIP Receiver
const CCIP_RECEIVER: &str = r#"
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

import "@chainlink/contracts/src/v0.8/ccip/interfaces/IAny2EVMMessageReceiver.sol";
import "@chainlink/contracts/src/v0.8/ccip/libraries/Client.sol";
import "@chainlink/contracts/src/v0.8/ccip/applications/CCIPReceiver.sol";

contract CrossChainReceiver is CCIPReceiver {
    // Message recu
    struct Message {
        uint64 sourceChainSelector;
        address sender;
        bytes data;
    }

    Message[] public messages;

    event MessageReceived(
        uint64 indexed sourceChainSelector,
        address sender,
        bytes data
    );

    constructor(address router) CCIPReceiver(router) {}

    // Callback CCIP
    function _ccipReceive(
        Client.Any2EVMMessage memory message
    ) internal override {
        uint64 sourceChainSelector = message.sourceChainSelector;
        address sender = abi.decode(message.sender, (address));
        bytes memory data = message.data;

        messages.push(Message({
            sourceChainSelector: sourceChainSelector,
            sender: sender,
            data: data
        }));

        emit MessageReceived(sourceChainSelector, sender, data);
    }

    function getMessages() external view returns (Message[] memory) {
        return messages;
    }
}
"#;

println!("Receiver CCIP:");
println!("{}", CCIP_RECEIVER);

Receiver CCIP:


// SPDX-License-Identifier: MIT


pragma solidity ^0.8.28;


import "@chainlink/contracts/src/v0.8/ccip/interfaces/IAny2EVMMessageReceiver.sol";


import "@chainlink/contracts/src/v0.8/ccip/libraries/Client.sol";


import "@chainlink/contracts/src/v0.8/ccip/applications/CCIPReceiver.sol";


... (28 outputs left out)

    function getMessages() external view returns (Message[] memory) {


        return messages;


    }


}


Contrat expéditeur CCIP permettant d'envoyer des messages et des tokens vers d'autres blockchains via le protocole Chainlink.

In [3]:
// Chainlink CCIP Sender
const CCIP_SENDER: &str = r#"
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

import "@chainlink/contracts/src/v0.8/ccip/interfaces/IRouterClient.sol";
import "@chainlink/contracts/src/v0.8/ccip/libraries/Client.sol";
import "@chainlink/contracts/src/v0.8/shared/access/ConfirmedOwner.sol";

contract CrossChainSender is ConfirmedOwner {
    IRouterClient private immutable router;

    event MessageSent(bytes32 messageId);

    constructor(address _router) ConfirmedOwner(msg.sender) {
        router = IRouterClient(_router);
    }

    // Envoyer un message cross-chain
    function sendMessage(
        uint64 destinationChainSelector,
        address receiver,
        bytes calldata data,
        address feeToken
    ) external payable returns (bytes32) {
        // Construire le message
        Client.EVM2AnyMessage memory message = Client.EVM2AnyMessage({
            receiver: abi.encode(receiver),
            data: data,
            tokenAmounts: new Client.EVMTokenAmount[](0),
            extraArgs: Client._argsToBytes(
                Client.EVMExtraArgsV1({
                    gasLimit: 200_000
                })
            ),
            feeToken: feeToken
        });

        // Calculer les frais
        uint256 fees = router.getFee(destinationChainSelector, message);
        require(msg.value >= fees, "Insufficient fee");

        // Envoyer via CCIP
        bytes32 messageId = router.ccipSend{value: fees}(
            destinationChainSelector,
            message
        );

        emit MessageSent(messageId);
        return messageId;
    }

    // Envoyer tokens + message
    function sendMessageWithToken(
        uint64 destinationChainSelector,
        address receiver,
        bytes calldata data,
        address token,
        uint256 amount,
        address feeToken
    ) external payable returns (bytes32) {
        // Transfer tokens to this contract
        IERC20(token).transferFrom(msg.sender, address(this), amount);
        IERC20(token).approve(address(router), amount);

        Client.EVMTokenAmount[] memory tokenAmounts = 
            new Client.EVMTokenAmount[](1);
        tokenAmounts[0] = Client.EVMTokenAmount({
            token: token,
            amount: amount
        });

        Client.EVM2AnyMessage memory message = Client.EVM2AnyMessage({
            receiver: abi.encode(receiver),
            data: data,
            tokenAmounts: tokenAmounts,
            extraArgs: Client._argsToBytes(
                Client.EVMExtraArgsV1({
                    gasLimit: 200_000
                })
            ),
            feeToken: feeToken
        });

        uint256 fees = router.getFee(destinationChainSelector, message);
        require(msg.value >= fees, "Insufficient fee");

        bytes32 messageId = router.ccipSend{value: fees}(
            destinationChainSelector,
            message
        );

        emit MessageSent(messageId);
        return messageId;
    }
}

interface IERC20 {
    function transferFrom(address, address, uint256) external returns (bool);
    function approve(address, uint256) external returns (bool);
}
"#;

println!("Sender CCIP:");
println!("{}", CCIP_SENDER);

Sender CCIP:


// SPDX-License-Identifier: MIT


pragma solidity ^0.8.28;


import "@chainlink/contracts/src/v0.8/ccip/interfaces/IRouterClient.sol";


import "@chainlink/contracts/src/v0.8/ccip/libraries/Client.sol";


import "@chainlink/contracts/src/v0.8/shared/access/ConfirmedOwner.sol";


... (75 outputs left out)

interface IERC20 {


    function transferFrom(address, address, uint256) external returns (bool);


    function approve(address, uint256) external returns (bool);


}


### Interpretation : Enjeux et Solutions Cross-Chain

L'interoperabilite entre blockchains presente des defis uniques que les contrats single-chain n'ont pas.

| Problème | Description | Impact sur les bridges |
|----------|-------------|----------------------|
| **Finalite** | Une transaction peut etre revertue même après etre incluse dans un bloc | Il faut attendre suffisamment de confirmations avant de considerer une transaction comme finale |
| **Securite** | Une attaque sur une chain peut affecter l'autre (reentrancy cross-chain) | Les bridges doivent etre extremement securises, car une breach peut drainer des fonds sur plusieurs chains |
| **Latence** | Le temps de confirmation varie entre les chains (secondes a minutes) | L'UX cross-chain est plus lent que les transactions single-chain |
| **Cout** | Gas sur les deux chains + frais de bridge | Les transferts cross-chain sont beaucoup plus chers que les transferts locaux |
| **Oracle** | Il faut un moyen de verifier l'etat d'une autre chain | Dépendance a des tiers (oracles, validators, guardians) |

**Points cles** :
1. **Lock & Mint** : Lock tokens A sur Ethereum, mint wrapped A' sur Polygon (le plus courant)
2. **Burn & Mint** : Burn wrapped A' sur Polygon, unlock A sur Ethereum (reverse opération)
3. **Liquidity Pools** : Utiliser des pools de liquidite sur chaque chaîne (ex: Hop Protocol, Stargate)
4. **Light Clients** : Verifier les headers de la blockchain source (ex: Optimism, Arbitrum use optimistic rollups)

**Protocoles majeurs** :
- **Chainlink CCIP** : Oracle-based messaging (securite maximale, latence moyenne)
- **LayerZero** : Ultra-light nodes (rapide, mais oracle + relayer = 2 points de confiance)
- **Axelar** : Gateway avec Proof-of-Stake (securite decentralisee, ecosystem Cosmos)
- **Wormhole** : Guardian network (19 guardians, integration Solana forte)
- **Hyperlane** : Permissionless (chacun peut operer un securateur, modèle economique)

> **Note technique** : Aucun bridge n'est parfait. Chaque protocol fait des trade-offs entre securite, decentralisation, rapidite et cout. Le choix depend du cas d'usage (transferts de grande valeur = securite maximale).

***

## 3. Chain Selectors

### Chain selectors CCIP

Source officielle : smartcontractkit/chain-selectors (selectors.yml, branche main)

| Chain | Selector |
|---|---|
| Ethereum Mainnet | 5009297550715157269 |
| Ethereum Sepolia | 16015286601757825753 |
| Arbitrum Mainnet | 4949039107694359620 |
| Arbitrum Sepolia | 3478487238524512106 |
| Optimism Mainnet | 3734403246176062136 |
| Optimism Sepolia | 5224473277236331295 |
| Polygon Mainnet | 4051577828743386545 |
| Polygon Mumbai | 12532609583862916517 |
| Avalanche Mainnet | 6433500567565415381 |
| Avalanche Fuji | 14767482510784806043 |
| Base Mainnet | 15971525489660198786 |
| Base Sepolia | 10344971235874465080 |

**ROUTER ADDRESSES (CCIP, chaines EVM) :**
- Sepolia : `0x0BF3dE8c5D3e8A2B34D2BEeB17ABfCeBaf363A59`
- Mainnet : `0x80226fc0Ee2b096224EeAc085Bb9a8cba1146f7D`

### Le receiver CCIP (Chainlink)

Ce contrat montre comment recevoir des messages cross-chain via Chainlink CCIP.

| Composant | Rôle | Details |
|-----------|------|---------|
| `CCIPReceiver` | Contrat de base abstrait | Implemente la reception CCIP et le contrôle du router |
| `_ccipReceive()` | Callback interne | Appele par `ccipReceive()`, que seul le router CCIP peut appeler |
| `messages[]` | Stockage des messages | Historique de tous les messages recus |
| `MessageReceived` event | Notification | Permet aux clients d'ecouter les arrivées |

**Points cles** :
1. Le contrat herite de `CCIPReceiver(router)` qui gere la logique de verification du router CCIP
2. `_ccipReceive()` est `internal virtual` : c'est `ccipReceive()` (fonction externe `onlyRouter`) qui l'appelle — seul le router CCIP est autorise a livrer un message
3. `message.sourceChainSelector` : identifie la chaîne d'ou vient le message (ex: Sepolia, Arbitrum)
4. `abi.decode(message.sender, (address))` : l'envoyeur est encode en bytes, il faut le decoder
5. `message.data` : contenu du message (peut etre n'importe quoi : string, struct, etc.)

**Workflow de reception** :
1. Un contrat Sender sur une autre chaîne appelle `ccipSend()` avec ce contrat comme destinataire
2. Les oracles Chainlink relayent le message vers cette chaîne
3. Le router CCIP sur cette chaîne appelle `_ccipReceive()` de ce contrat
4. Le contrat stocke le message et emet un événement `MessageReceived`

**Cas d'usage** :
- Bridge de tokens : recevoir un message "Lock" et mint des tokens representatifs
- Messagerie : recevoir des données d'une dApp sur une autre chaîne
- Oracle multi-chain : agreger des données de plusieurs chains dans un seul contrat

> **Note technique** : `_ccipReceive()` doit etre marque `internal` et `virtual`. C'est le **router CCIP**, via `ccipReceive()` (`onlyRouter`), qui appelle cette fonction : un appel direct par un utilisateur est rejete.

### Exercice : Simulateur de messagerie CCIP

Le protocole CCIP route les messages entre blockchains via des chain selectors et des routers. Completez la classe `CCIPSimulator` qui simule le flux complet d'envoi et de reception de messages cross-chain.

**Objectifs** :
1. Implementer `send_message` qui enregistre un message avec ses metadonnees (source, destination, données, frais)
2. Implementer `receive_messages` qui recupere et decode tous les messages pour un destinataire donne
3. Calculer les frais de messagerie en fonction de la taille des données et de la distance entre chains

**Indices** :
- Les frais dependent de la taille des données (en octets) et du chain selector de destination
- Chaque message a un `messageId` unique (hash du contenu)
- Un message ne peut etre recu que si la chaîne source est enregistree dans le routeur

In [4]:
// Exercice : Simulateur de messagerie CCIP

struct ChainConfig { name: &'static str, selector: u64, base_fee: f64 }

const CHAIN_CONFIG: [ChainConfig; 4] = [
    ChainConfig { name: "ethereum_sepolia", selector: 16015286601757825753, base_fee: 0.001 },
    ChainConfig { name: "arbitrum_sepolia", selector: 3478487238524512106, base_fee: 0.0005 },
    ChainConfig { name: "polygon_mumbai", selector: 12532609583862916517, base_fee: 0.0003 },
    ChainConfig { name: "optimism_sepolia", selector: 5224473277236331295, base_fee: 0.0004 },
];

struct Message { message_id: String, fees: f64, status: &'static str }

struct CCIPSimulator {
    messages: Vec<Message>,                    // liste des messages envoyes
    received: Vec<(String, Vec<Message>)>,     // address -> liste des messages recus
}

impl CCIPSimulator {
    fn new() -> Self {
        CCIPSimulator { messages: vec![], received: vec![] }
    }

    fn send_message(&mut self, source_chain: &str, dest_chain: &str,
                    sender: &str, receiver: &str, data: &str) -> Option<Message> {
        // TODO etudiant : implementez send_message
        // Etape 1 : verifier que source_chain et dest_chain sont dans CHAIN_CONFIG
        // Etape 2 : calculer le messageId (hash des metadonnees)
        // Etape 3 : calculer les frais (base_fee * taille_donnees_en_octets)
        // Etape 4 : enregistrer le message dans self.messages
        // Etape 5 : retourner un Message avec messageId, fees, status
        None // TODO etudiant
    }

    fn receive_messages(&mut self, receiver: &str, dest_chain: &str) -> Vec<Message> {
        // TODO etudiant : implementez receive_messages
        // Etape 1 : filtrer self.messages pour ceux avec receiver et dest_chain
        // Etape 2 : ajouter chaque message dans self.received[receiver]
        // Etape 3 : retourner la liste des messages recus
        vec![] // TODO etudiant
    }
}

// Test du simulateur
let mut sim = CCIPSimulator::new();
let result = sim.send_message(
    "ethereum_sepolia", "arbitrum_sepolia",
    "0xSender123", "0xReceiver456", "Hello from Ethereum!"
);
if let Some(r) = result {
    println!("Message envoye : {}", r.message_id);
    println!("Frais : {} ETH", r.fees);
    let msgs = sim.receive_messages("0xReceiver456", "arbitrum_sepolia");
    println!("Messages recus : {}", msgs.len());
} else {
    println!("Message non envoye (chain inconnue)");
}
println!("Exercice a completer : CCIPSimulator");

Message non envoye (chain inconnue)


Exercice a completer : CCIPSimulator


***

## 4. Securite Cross-Chain

### Securite cross-chain

1. **REENTRANCY CROSS-CHAIN**
   - Une attaque sur une chain peut affecter l'autre
   - Solution : Verifier la finalite avant execution
2. **DOUBLE-SPEND**
   - Meme tokens utilises sur plusieurs chains
   - Solution : Lock/Burn atomique
3. **ORACLE MANIPULATION**
   - Fausse information sur l'etat d'une chain
   - Solution : Multiples oracles, verification
4. **BRIDGE EXPLOITS**
   - Bugs dans les contrats de bridge
   - Solution : Audits, bug bounties, limits
5. **GOVERNANCE ATTACKS**
   - Manipulation des votes cross-chain
   - Solution : Snapshot voting, delay

**Best practices :**
- Toujours verifier la finalite (confirmations)
- Rate limits sur les transferts
- Emergency pause mechanism
- Audits reguliers
- Monitoring et alertes

### Exercice : Analyse de risque d'un bridge

Avant de deployer un bridge en production, il est essentiel d'identifier et de quantifier les risques. Completez la fonction `assess_bridge_risk` qui analyse les paramètres d'un bridge et retourne un score de risque.

**Objectifs** :
1. Evaluer le risque lie au nombre de validateurs et au seuil de signature
2. Verifier la presence de protections (rate limit, emergency pause, timelock)
3. Calculer un score de risque composite (0-100)

**Indices** :
- Un seuil M-of-N avec N < 5 est considere comme risque eleve
- L'absence de rate limit ou d'emergency pause augmente le score de 20 points chacun
- Le score final est plafonne a 100

In [5]:
// Exercice : Analyse de risque d'un bridge

struct Risque { score: i64, recommendations: Vec<String> }

fn assess_bridge_risk(num_validators: i64, threshold: i64,
                      has_rate_limit: bool, has_emergency_pause: bool,
                      has_timelock: bool, max_daily_transfer: i64) -> Risque {
    // TODO etudiant : implementez assess_bridge_risk
    // Etape 1 : calculer un score de base selon N et M (plus N est grand, plus c'est sur)
    // Etape 2 : ajouter des penalties si les protections manquent
    // Etape 3 : ajuster selon le montant quotidien
    // Etape 4 : generer des recommandations specifiques
    Risque { score: 0, recommendations: vec![] } // TODO etudiant
}

// Test avec un bridge hypothetique
let r = assess_bridge_risk(
    3, 2,
    false, true,
    false, 1_000_000
);
println!("Score de risque : {}/100", r.score);
for rec in &r.recommendations {
    println!("  - {}", rec);
}
println!("Exercice a completer : assess_bridge_risk");

Score de risque : 0/100


Exercice a completer : assess_bridge_risk


### Interpretation : Sender CCIP Chainlink

Ce contrat montre comment envoyer des messages et des tokens vers d'autres blockchains via Chainlink CCIP.

| Fonction | Opération | Contenu du message |
|----------|-----------|-------------------|
| `sendMessage` | Message simple (text, données) | `data` uniquement, pas de tokens |
| `sendMessageWithToken` | Message + transfert de tokens ERC-20 | `data` + `tokenAmounts[]` (1 token dans l'exemple) |

**Points cles** :
1. `destinationChainSelector` : identifiant unique de la chaîne cible (ex: Sepolia = 16015286601757825753)
2. `Client.EVM2AnyMessage` : structure CCIP qui contient toutes les infos necessaires au routage
3. `router.getFee()` : calcule les frais CCIP en fonction de la destination et de la taille du message
4. `router.ccipSend{value: fees}()` : envoie le message CCIP (paye les frais avec ETH via `{value: fees}`)
5. `extraArgs` : `Client._argsToBytes(Client.EVMExtraArgsV1({ gasLimit: 200_000 }))` — sans `extraArgs`, le routeur applique sa valeur par defaut (200 000 gas pour le callback) ; `EVMExtraArgsV2` ajoute `allowOutOfOrderExecution`, dont la valeur attendue est imposee par certaines chaînes

**Workflow d'envoi avec tokens** :
1. L'utilisateur appelle `sendMessageWithToken()` avec les tokens transferes (`transferFrom`)
2. Le contrat approuve le router CCIP pour depenser les tokens (`approve`)
3. Le router CCIP lock les tokens et créé un message avec `tokenAmounts[]`
4. Les oracles Chainlink relayent le message et debloquent les tokens sur la destination

**Securite** :
- `ConfirmedOwner` : permet de **transferer la propriete** du contrat (`transferOwnership`) ; le router, lui, est `immutable` — fixe a la construction, sans aucun setter
- `require(msg.value >= fees)` : verifie que l'envoyeur a paye assez de frais — mais l'**excedent** reste dans le contrat, qui n'a **aucune fonction de retrait** : envoyer exactement les frais
- Les tokens sont transferes au contrat AVANT l'envoi (pas de reentrancy possible)

> **Note technique** : CCIP supporte l'envoi de multiples tokens dans un seul message (`tokenAmounts[]`). L'exemple n'en montre qu'un, mais on pourrait envoyer 3+ tokens en un appel.

***

## 5. Resume

### Resume cross-chain

| Protocol | Type | Securite |
|---|---|---|
| Chainlink CCIP | Oracle-based | Oracles CCIP |
| LayerZero | Ultra-light | Oracle + Relayer |
| Axelar | Gateway | Proof-of-Stake |
| Wormhole | Guardians | 13/19 (quorum) |
| Hyperlane | Permissionless | Economic security |

**Quand utiliser :**
- **CCIP** : Pour securite maximale, tokens transfers
- **LayerZero** : Pour rapidite, omnichain dApps
- **Axelar** : Pour ecosystem cosmos + EVM
- **Wormhole** : Pour Solana integration

***

**Notebook suivant** : [SC-24-Testnet-Deploy-Python](SC-24-Testnet-Deploy-Python.ipynb) | [jumeau Python](SC-23-Cross-Chain-Python.ipynb)

### Interpretation : Chain Selectors Chainlink CCIP

Les chain selectors sont des identifiants uniques que CCIP utilise pour router les messages entre les blockchains.

| Chaîne | Selector | Usage |
|--------|----------|------|
| **Ethereum Sepolia** | 16015286601757825753 | Testnet Ethereum (développement) |
| **Ethereum Mainnet** | 5009297550715157269 | Production Ethereum |
| **Arbitrum Sepolia** | 3478487238524512106 | Testnet L2 |
| **Polygon Mumbai** | 12532609583862916517 | Testnet Polygon |
| **Base Sepolia** | 10344971235874465080 | Testnet Base (Coinbase) |

**Points cles** :
1. Les selectors sont des nombres `uint64` uniques generes par Chainlink pour chaque chaîne supportee
2. Le router est le point d'entree unique pour envoyer des messages CCIP sur une chaîne donnee
3. L'adresse du router Sepolia (`0x0BF3dE8...`) est utilisee pour construire les contrats Sender/Receiver
4. Le router Mainnet (`0x80226fc...`) est différent : toujours utiliser le bon router pour la bonne chaîne

**Workflow CCIP** :
1. Sur la source : `sender.sendMessage(destinationSelector, receiver, data)` → paye des frais CCIP
2. Les oracles Chainlink detectent le message et le relayent vers la destination
3. Sur la destination : `receiver._ccipReceive(message)` est appele automatiquement
4. Les frais sont payes en `feeToken` (souvent LINK ou native token de la chaîne)

**Gas economise** : CCIP gere la complexite du routing cross-chain. Sans CCIP, il faudrait implementer un reseau d'oracles et de relayers soi-même.

> **Note technique** : un selector est attribue par Chainlink à une chaîne et lui reste ; la **source de verite** reste le registre officiel `smartcontractkit/chain-selectors` (`selectors.yml`), a relire avant tout deploiement — le registre evolue (nouvelles chaînes, deprecations).

## Exemple guide : Bridge Token Cross-Chain — Solution etudiante (Lucas Demuliere & Joanne Jabbour, TP 2026)

Implementation complete d'un bridge **Lock & Mint** entre deux chaînes simulees, avec **protection ECDSA** contre les replay attacks et **simulation Rust** du flux cross-chain.

### Objectifs couverts
1. Comprendre le pattern Lock & Mint
2. Implementer un bridge securise
3. Gerer les cas d'erreur et reverts
4. Tester le flux complet

### Architecture

| Cote source | Cote destination |
|-------------|------------------|
| `TokenBridge` : `lock()` verrouille, `unlock()` libere sur preuve ECDSA | `WrappedToken` : `mint()` sur Lock detecte, `burn()` pour le retour |
| Validator unique (ECDSA) | Bridge autorise uniquement (onlyBridge) |

### Securite implementee
- **Nonce unique** par opération (protection replay)
- **chainId dans le hash** (protection cross-chain replay)
- **Signature ECDSA verifiee** via OpenZeppelin `ECDSA.recover` (contrat Solidity ; la simulation Rust en reproduit le principe avec un condensat)
- **Double mapping** source/destination (un nonce valide d'un cote n'est pas valide de l'autre)

In [6]:
// Exercice a completer : Bridge multi-validateurs (M-of-N)
// Etudiant : etendez le simulateur Lock & Mint (TokenBridgeSimulator, presente plus bas dans ce carnet)

use std::collections::HashSet;

struct MultiValidatorBridge {
    validators: HashSet<String>,
    threshold: usize,
}

impl MultiValidatorBridge {
    fn new(validators: HashSet<String>, threshold: usize) -> Self {
        // TODO etudiant : initialiser validators, threshold, etat locked/wrapped/nonces
        MultiValidatorBridge { validators, threshold } // TODO etudiant : completer l'initialisation
    }

    fn lock(&mut self, user: &str, amount: i64) -> i64 {
        // Etape 1 : incrementer locked[user]
        // Etape 2 : generer un nonce unique
        // Etape 3 : retourner le nonce
        -1 // TODO etudiant : placeholder, a remplacer
    }

    fn unlock(&mut self, user: &str, amount: i64, nonce: i64, signing_validators: &[String]) -> Result<(), String> {
        // TODO etudiant :
        // - Verifier qu'aucun validator ne signe deux fois (doublon -> refus)
        // - Verifier que tous les signing_validators sont dans self.validators
        // - Verifier que le nombre de signataires distincts est >= self.threshold
        // - Verifier que nonce pas deja utilise
        // - Verifier que locked[user] >= amount
        // - Appliquer l'unlock
        // Indice : les signatures arrivent en LISTE (un meme validator peut
        // signer deux fois) : detecter les doublons avant de compter.
        // if signing_validators.len() != HashSet::from_iter(signing_validators.iter().cloned()).len() { return Err(...); }
        // let valid_signers: HashSet<String> = ...;
        // if valid_signers.len() < self.threshold { return Err(...); }
        Ok(()) // TODO etudiant
    }

    fn mint(&mut self, user: &str, amount: i64, nonce: i64) -> Result<(), String> {
        // TODO etudiant : protection replay par nonce (un nonce ne peut servir qu'une fois)
        Ok(()) // TODO etudiant
    }

    fn burn(&mut self, user: &str, amount: i64) -> i64 {
        // TODO etudiant : verifier balance, decrementer, retourner nonce
        -1 // TODO etudiant : placeholder
    }

    fn rotate_validator(&mut self, old_validator: &str, new_validator: &str, signing_validators: &[String]) -> Result<(), String> {
        // TODO etudiant :
        // - Verifier que old_validator in self.validators
        // - Verifier que new_validator not in self.validators
        // - Verifier que len(set(signing_validators) & self.validators) >= threshold
        // - Appliquer la rotation
        Ok(()) // TODO etudiant
    }
}

// Tests preliminaires (l'estudiant doit les completer)
let validators: HashSet<String> = (0..13).map(|i| format!("validator_{}", i)).collect();
println!("Bridge multi-validateurs initialise : {} validators, seuil 9", validators.len());
let _bridge = MultiValidatorBridge::new(validators, 9);
println!("Exercice a completer : MultiValidatorBridge M-of-N");

Bridge multi-validateurs initialise : 13 validators, seuil 9


Exercice a completer : MultiValidatorBridge M-of-N


## Exercice : Bridge multi-validateurs (M-of-N)

Le carnet presente un bridge **Lock & Mint** a **validateur unique** (struct `TokenBridgeSimulator`, plus bas dans ce carnet) : un seul validator signe les liberations, c'est un point de defaillance — s'il est compromis, tous les fonds verrouilles sont en danger. Les bridges industriels (Wormhole, Axelar, LayerZero) utilisent un **ensemble de N validators** avec un seuil **M-of-N** (ex: 13 validateurs, seuil 9).

### Objectifs
1. Etendre le simulateur pour gerer plusieurs validators
2. Implementer la verification de seuil (M signatures requises pour unlock)
3. Gerer la rotation des validators (join/leave) avec accord du seuil
4. Tester la robustesse face a un validator compromis

**Indices :**
- Stocker `validators: HashSet<String>` et un seuil `threshold: usize` (M validators sur N)
- Pour `unlock`, accepter un `Vec<String> signing_validators` (les validateurs ayant signe, **dans l'ordre d'arrivee**) : un `HashSet` dedupliquerait les doublons **avant** tout contrôle, et un validator qui signe deux fois serait indiscernable d'un validator honnete. Verifier d'abord l'absence de doublon, puis que le nombre de signataires distincts est >= threshold
- Le hash signe doit inclure l'**adresse du validator** pour empecher qu'une signature d'un validator soit reusee pour un autre
- `rotate_validator(old, new, signing_validators)` requiert le meme seuil que unlock (changement de configuration = même seuil)

### Cas de test attendus
1. **Unlock valide** : 9 validators sur 13 signent -> unlock reussit
2. **Unlock rejete** : 8 validators sur 13 signent (sous le seuil) -> revert
3. **Validator compromis** : 1 validator signe deux fois le même nonce -> revert (doublon detecte)
4. **Rotation** : remplacement d'un validator avec accord du seuil -> reussit

### Analyse : Securite et Limites d'un Bridge Cross-Chain
*Reponses proposees par Lucas Demuliere & Joanne Jabbour (TP 2026)*

**Q1 — Comment empecher les attaques par rejeu (replay attacks) ?**

| Mécanisme | Implementation | Garantie |
|-----------|---------------|---------|
| **Nonce unique** | `mapping(uint256 => bool) usedNonces` | Chaque opération ne peut etre executee qu'une fois |
| **ChainId dans le hash** | `keccak256(abi.encodePacked(user, amount, nonce, block.chainid))` | Empeche la reutilisation de la signature sur une autre chain |
| **Double mapping** | Un mapping par chain (source et destination) | Un nonce valide sur la source n'est pas valide sur la destination |

> Sans le `chainid` dans le hash signe, une signature valide sur Sepolia pourrait etre rejouee sur Mainnet et drainer les fonds.

***

**Q2 — Quelle est la latence minimale pour une transaction cross-chain ?**

```
Confirmation source  (12 blocs Ethereum ~ 2 min)
     + Detection oracle/relayer            (~30 sec)
     + Transmission du message             (~variable)
     + Confirmation destination            (~2-12 sec selon L2)
     -----------------------------------------------
     TOTAL minimal : ~3-5 minutes  (Chainlink CCIP Sepolia)
                     ~20-30 min    (Mainnet haute securite)
```

- Les L2 (Arbitrum, Optimism) recoivent plus vite, mais la finalite sur L1 reste lente
- Chainlink CCIP attend plusieurs confirmations avant de relayer pour eviter les reorgs
- Les bridges "optimistic" (Hop, Across) sont plus rapides mais acceptent un risque de reorg

***

**Q3 — Comment gerer les echecs sur la chaîne destination ?**

| Scénario | Problème | Solution |
|----------|----------|---------|
| Contrat destination en pause | Le mint echoue silencieusement | Mécanisme de retry + monitoring oracle |
| Gas insuffisant | La transaction destination est droppee | Fournir un surplus de gas + retry automatique CCIP |
| Contrat destination inexistant | Tokens bloques indefiniment sur source | Whitelist des destinations autorisees avant envoi |
| Reorg sur source | Lock non finalisee mais Mint effectue | Attendre N confirmations avant de relayer (CCIP : 7 blocs) |

**Pattern recommande** : circuit breaker avec emergency pause + timelock de 48h sur les changements de config du validator.

In [7]:
// Simulation Rust du flux cross-chain (Lock -> Mint -> Burn -> Unlock)
// Solution proposee par Lucas Demuliere & Joanne Jabbour (TP 2026)

use std::collections::{HashMap, HashSet};

// SHA-256 (FIPS 180-4) en Rust std seul -- bloc a recopier tel quel dans chaque cellule qui hache.
// Verifie contre hashlib.sha256 (CPython) par le script de rejeu de la page.
fn sha256(data: &[u8]) -> [u8; 32] {
    const K: [u32; 64] = [
        0x428a2f98, 0x71374491, 0xb5c0fbcf, 0xe9b5dba5, 0x3956c25b, 0x59f111f1, 0x923f82a4, 0xab1c5ed5,
        0xd807aa98, 0x12835b01, 0x243185be, 0x550c7dc3, 0x72be5d74, 0x80deb1fe, 0x9bdc06a7, 0xc19bf174,
        0xe49b69c1, 0xefbe4786, 0x0fc19dc6, 0x240ca1cc, 0x2de92c6f, 0x4a7484aa, 0x5cb0a9dc, 0x76f988da,
        0x983e5152, 0xa831c66d, 0xb00327c8, 0xbf597fc7, 0xc6e00bf3, 0xd5a79147, 0x06ca6351, 0x14292967,
        0x27b70a85, 0x2e1b2138, 0x4d2c6dfc, 0x53380d13, 0x650a7354, 0x766a0abb, 0x81c2c92e, 0x92722c85,
        0xa2bfe8a1, 0xa81a664b, 0xc24b8b70, 0xc76c51a3, 0xd192e819, 0xd6990624, 0xf40e3585, 0x106aa070,
        0x19a4c116, 0x1e376c08, 0x2748774c, 0x34b0bcb5, 0x391c0cb3, 0x4ed8aa4a, 0x5b9cca4f, 0x682e6ff3,
        0x748f82ee, 0x78a5636f, 0x84c87814, 0x8cc70208, 0x90befffa, 0xa4506ceb, 0xbef9a3f7, 0xc67178f2,
    ];
    let mut h: [u32; 8] = [0x6a09e667, 0xbb67ae85, 0x3c6ef372, 0xa54ff53a, 0x510e527f, 0x9b05688c, 0x1f83d9ab, 0x5be0cd19];
    let mut msg = data.to_vec();
    let bit_len = (data.len() as u64).wrapping_mul(8);
    msg.push(0x80);
    while msg.len() % 64 != 56 {
        msg.push(0);
    }
    msg.extend_from_slice(&bit_len.to_be_bytes());
    for chunk in msg.chunks(64) {
        let mut w = [0u32; 64];
        for i in 0..16 {
            w[i] = u32::from_be_bytes([chunk[4 * i], chunk[4 * i + 1], chunk[4 * i + 2], chunk[4 * i + 3]]);
        }
        for i in 16..64 {
            let s0 = w[i - 15].rotate_right(7) ^ w[i - 15].rotate_right(18) ^ (w[i - 15] >> 3);
            let s1 = w[i - 2].rotate_right(17) ^ w[i - 2].rotate_right(19) ^ (w[i - 2] >> 10);
            w[i] = w[i - 16].wrapping_add(s0).wrapping_add(w[i - 7]).wrapping_add(s1);
        }
        let mut v = h;
        for i in 0..64 {
            let s1 = v[4].rotate_right(6) ^ v[4].rotate_right(11) ^ v[4].rotate_right(25);
            let ch = (v[4] & v[5]) ^ (!v[4] & v[6]);
            let t1 = v[7].wrapping_add(s1).wrapping_add(ch).wrapping_add(K[i]).wrapping_add(w[i]);
            let s0 = v[0].rotate_right(2) ^ v[0].rotate_right(13) ^ v[0].rotate_right(22);
            let maj = (v[0] & v[1]) ^ (v[0] & v[2]) ^ (v[1] & v[2]);
            let t2 = s0.wrapping_add(maj);
            v = [t1.wrapping_add(t2), v[0], v[1], v[2], v[3].wrapping_add(t1), v[4], v[5], v[6]];
        }
        for i in 0..8 {
            h[i] = h[i].wrapping_add(v[i]);
        }
    }
    let mut out = [0u8; 32];
    for i in 0..8 {
        out[4 * i..4 * i + 4].copy_from_slice(&h[i].to_be_bytes());
    }
    out
}

// Equivalent de hashlib.sha256(data).hexdigest() : 64 chiffres hexadecimaux minuscules.
fn sha256_hex(data: &[u8]) -> String {
    sha256(data).iter().map(|b| format!("{:02x}", b)).collect()
}

// Secret partage du validateur unique de la simulation.
// Ce n'est PAS une cle ECDSA : la simulation montre le mecanisme
// (un tiers signe, le bridge recalcule et compare) ; la verification
// ECDSA reelle est dans le contrat Solidity de la cellule suivante.
const VALIDATOR_SECRET: &str = "validator_secret_demo";

fn sign(secret: &str, nonce: u64, amount: i64) -> String {
    sha256_hex(format!("{}:{}:{}", secret, nonce, amount).as_bytes())
}

fn py_dict(m: &[(String, i64)]) -> String {
    let items: Vec<String> = m.iter().map(|(k, v)| format!("'{}': {}", k, v)).collect();
    format!("{{{}}}", items.join(", "))
}

fn get(m: &[(String, i64)], key: &str) -> i64 {
    m.iter().find(|(k, _)| k == key).map(|(_, v)| *v).unwrap_or(0)
}

fn add(m: &mut Vec<(String, i64)>, key: &str, delta: i64) {
    if let Some(entry) = m.iter_mut().find(|(k, _)| k == key) {
        entry.1 += delta;
    } else {
        m.push((key.to_string(), delta));
    }
}

struct TokenBridgeSimulator {
    locked: Vec<(String, i64)>,
    wrapped_balances: Vec<(String, i64)>,
    used_nonces_source: HashSet<u64>,
    used_nonces_dest: HashSet<u64>,
    // nonce de burn -> montant effectivement brule
    burned_amounts: HashMap<u64, i64>,
    nonce: u64,
    total_supply: i64,
}

impl TokenBridgeSimulator {
    fn new() -> Self {
        TokenBridgeSimulator {
            locked: vec![],
            wrapped_balances: vec![],
            used_nonces_source: HashSet::new(),
            used_nonces_dest: HashSet::new(),
            burned_amounts: HashMap::new(),
            nonce: 0,
            total_supply: 0,
        }
    }

    fn lock(&mut self, user: &str, amount: i64) -> u64 {
        add(&mut self.locked, user, amount);
        let n = self.nonce;
        self.nonce += 1;
        println!("[SOURCE] Lock   : {} verrouille {} ETH  (nonce={})", user, amount, n);
        n
    }

    fn mint(&mut self, user: &str, amount: i64, nonce: u64) -> Result<(), String> {
        if self.used_nonces_dest.contains(&nonce) {
            return Err(format!("Replay detecte ! nonce {} deja utilise sur destination", nonce));
        }
        self.used_nonces_dest.insert(nonce);
        add(&mut self.wrapped_balances, user, amount);
        self.total_supply += amount;
        println!("[DEST]   Mint   : {} recoit {} WETHb   (nonce={})", user, amount, nonce);
        Ok(())
    }

    fn burn(&mut self, user: &str, amount: i64) -> Result<u64, String> {
        if get(&self.wrapped_balances, user) < amount {
            return Err("Balance insuffisante".to_string());
        }
        add(&mut self.wrapped_balances, user, -amount);
        self.total_supply -= amount;
        let n = self.nonce;
        self.nonce += 1;
        // Le montant brule est memorise : c'est lui que l'unlock devra rendre.
        self.burned_amounts.insert(n, amount);
        println!("[DEST]   Burn   : {} retourne {} WETHb  (nonce={})", user, amount, n);
        Ok(n)
    }

    fn unlock(&mut self, user: &str, amount: i64, nonce: u64, signature: &str) -> Result<(), String> {
        // Trois controles, dans cet ordre :
        // 1. la signature doit correspondre au couple (nonce, montant) ;
        // 2. le nonce ne doit pas avoir deja servi sur la source ;
        // 3. le montant demande doit etre exactement celui qui a ete brule.
        let expected = sign(VALIDATOR_SECRET, nonce, amount);
        if signature != expected {
            return Err("Rejete : signature invalide pour ce nonce et ce montant.".to_string());
        }
        if self.used_nonces_source.contains(&nonce) {
            return Err(format!("Replay detecte ! nonce {} deja utilise sur source", nonce));
        }
        let burned = self.burned_amounts.get(&nonce).copied();
        match burned {
            None => return Err(format!("Rejete : aucun burn ne correspond au nonce {}.", nonce)),
            Some(b) if b != amount => return Err(format!("Rejete : {} WETHb ont ete brules, on ne peut liberer que {}.", b, b)),
            Some(_) => {}
        }
        if get(&self.locked, user) < amount {
            return Err("Balance verrouillee insuffisante".to_string());
        }
        self.used_nonces_source.insert(nonce);
        add(&mut self.locked, user, -amount);
        println!("[SOURCE] Unlock : {} recoit {} ETH orig (nonce={})", user, amount, nonce);
        Ok(())
    }

    fn state(&self) {
        println!("  locked={}  wrapped={}  supply={}", py_dict(&self.locked), py_dict(&self.wrapped_balances), self.total_supply);
    }
}

let bar = "=".repeat(55);
println!("{}", bar);
println!("TEST 1 : Round-trip complet (Lock -> Mint -> Burn -> Unlock)");
println!("{}", bar);
let mut b = TokenBridgeSimulator::new();
let n1 = b.lock("Alice", 100);
b.mint("Alice", 100, n1).unwrap();
b.state();
let n2 = b.burn("Alice", 100).unwrap();
b.unlock("Alice", 100, n2, &sign(VALIDATOR_SECRET, n2, 100)).unwrap();
b.state();

println!("\n{}", bar);
println!("TEST 2 : Replay attack (reutilisation du meme nonce)");
println!("{}", bar);
let mut b2 = TokenBridgeSimulator::new();
let n = b2.lock("Bob", 50);
b2.mint("Bob", 50, n).unwrap();
println!("Tentative de second mint avec le meme nonce...");
match b2.mint("Bob", 50, n) { // Doit echouer
    Ok(()) => println!("ERREUR : Replay non detecte !"),
    Err(e) => println!("Bloque : {}", e),
}

println!("\n{}", bar);
println!("TEST 3 : Faux unlock (signature invalide)");
println!("{}", bar);
let mut b3 = TokenBridgeSimulator::new();
let n = b3.lock("Charlie", 75);
println!("Tentative d'unlock avec une signature qui ne correspond pas...");
match b3.unlock("Charlie", 75, n, "signature_invalide") {
    Ok(()) => println!("ERREUR : Faux unlock accepte !"),
    Err(e) => println!("Bloque : {}", e),
}

println!("\n{}", bar);
println!("TEST 4 : Invariants (total_supply == somme wrapped_balances)");
println!("{}", bar);
let mut b4 = TokenBridgeSimulator::new();
b4.lock("Dave", 200);
b4.mint("Dave", 200, 0).unwrap();
b4.lock("Eve", 50);
b4.mint("Eve", 50, 1).unwrap();
let somme: i64 = b4.wrapped_balances.iter().map(|(_, v)| *v).sum();
assert!(b4.total_supply == somme);
println!("OK : total_supply ({}) == somme wrapped ({})", b4.total_supply, somme);

println!("\n{}", bar);
println!("TEST 5 : Invariant economique (le montant libere == le montant brule)");
println!("{}", bar);
let mut b5 = TokenBridgeSimulator::new();
let n = b5.lock("Frank", 100);
b5.mint("Frank", 100, n).unwrap();
let nb = b5.burn("Frank", 50).unwrap(); // seuls 50 WETHb sont brules
println!("Tentative d'unlock de 100 ETH alors que 50 WETHb seulement ont ete brules...");
match b5.unlock("Frank", 100, nb, &sign(VALIDATOR_SECRET, nb, 100)) {
    Ok(()) => println!("ERREUR : montant incoherent accepte !"),
    Err(e) => println!("Bloque : {}", e),
}
println!("Unlock du montant reellement brule (50) :");
b5.unlock("Frank", 50, nb, &sign(VALIDATOR_SECRET, nb, 50)).unwrap();
b5.state();
println!("Tous les tests du bridge passent avec succes.");

TEST 1 : Round-trip complet (Lock -> Mint -> Burn -> Unlock)


[SOURCE] Lock   : Alice verrouille 100 ETH  (nonce=0)


[DEST]   Mint   : Alice recoit 100 WETHb   (nonce=0)


  locked={'Alice': 100}  wrapped={'Alice': 100}  supply=100


[DEST]   Burn   : Alice retourne 100 WETHb  (nonce=1)


[SOURCE] Unlock : Alice recoit 100 ETH orig (nonce=1)


... (22 outputs left out)

Unlock du montant reellement brule (50) :


[SOURCE] Unlock : Frank recoit 50 ETH orig (nonce=1)


  locked={'Frank': 50}  wrapped={'Frank': 50}  supply=50


Tous les tests du bridge passent avec succes.


### Implementation Solidity du Bridge Lock & Mint

La simulation Rust a valide la logique du pattern Lock & Mint : protection replay, verification de signature **simulee** (le bridge recalcule le condensat signe et le compare a celui qu'il recoit — ce n'est pas de l'ECDSA) et invariants de supply. La verification ECDSA reelle est dans la cellule Solidity suivante, avec **TokenBridge** (chaîne source : lock/unlock avec verification ECDSA) et **WrappedToken** (chaîne destination : mint/burn avec protection `onlyBridge`). Les contrats utilisent `block.chainid` dans le hash signe pour empecher le cross-chain replay.

In [8]:
// Solution : Bridge Token Cross-Chain (pattern Lock & Mint)
// Solution proposee par Lucas Demuliere & Joanne Jabbour (TP 2026)

const TOKEN_BRIDGE: &str = r#"
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

import "@openzeppelin/contracts/utils/cryptography/ECDSA.sol";
import "@openzeppelin/contracts/utils/cryptography/MessageHashUtils.sol";

/// @title TokenBridge - Chaine source (Lock & Unlock)
contract TokenBridge {
    using ECDSA for bytes32;

    mapping(address => uint256) public locked;      // user => montant verrouille
    mapping(uint256 => bool)    public usedNonces;  // protection replay
    address public immutable validator;
    uint256 public nonce;

    event Lock(address indexed user, uint256 amount, uint256 nonce);
    event Unlock(address indexed user, uint256 amount, uint256 nonce);

    constructor(address _validator) {
        validator = _validator;
    }

    /// Verrouille des ETH natifs et emet Lock pour que le relayer detecte
    function lock() external payable {
        require(msg.value > 0, "Amount must be > 0");
        locked[msg.sender] += msg.value;
        emit Lock(msg.sender, msg.value, nonce++);
    }

    /// Deverrouille les tokens - appele par le validator apres Burn sur dest
    function unlock(
        address user,
        uint256 amount,
        uint256 _nonce,
        bytes calldata signature
    ) external {
        require(!usedNonces[_nonce],          "Nonce already used");
        require(locked[user] >= amount,       "Insufficient locked balance");
        require(
            validateSignature(user, amount, _nonce, signature),
            "Invalid validator signature"
        );

        usedNonces[_nonce] = true;
        locked[user] -= amount;

        (bool ok,) = payable(user).call{value: amount}("");
        require(ok, "Transfer failed");

        emit Unlock(user, amount, _nonce);
    }

    /// Verifie la signature ECDSA du validator autorise
    /// Hash = keccak256(user, amount, nonce, chainId) - empeche le replay cross-chain
    function validateSignature(
        address user,
        uint256 amount,
        uint256 _nonce,
        bytes memory signature
    ) public view returns (bool) {
        bytes32 hash    = keccak256(abi.encodePacked(user, amount, _nonce, block.chainid));
        bytes32 ethHash = MessageHashUtils.toEthSignedMessageHash(hash);
        return ECDSA.recover(ethHash, signature) == validator;
    }

    receive() external payable {}
}
"#;

const WRAPPED_TOKEN: &str = r#"
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

/// @title WrappedToken - Chaine destination (Mint & Burn)
contract WrappedToken {
    string  public constant name     = "Wrapped ETH Bridge";
    string  public constant symbol   = "WETHb";
    uint8   public constant decimals = 18;

    mapping(address => uint256) public balanceOf;
    mapping(uint256 => bool)    public mintedNonces; // protection replay
    uint256 public totalSupply;
    address public immutable bridge;

    event Transfer(address indexed from, address indexed to, uint256 amount);
    event Mint(address indexed to, uint256 amount, uint256 nonce);
    event Burn(address indexed from, uint256 amount);

    modifier onlyBridge() {
        require(msg.sender == bridge, "Only bridge");
        _;
    }

    constructor(address _bridge) {
        bridge = _bridge;
    }

    /// Mint appele par le bridge quand evenement Lock detecte sur la source
    function mint(address user, uint256 amount, uint256 _nonce) external onlyBridge {
        require(!mintedNonces[_nonce], "Nonce already minted");
        mintedNonces[_nonce] = true;
        balanceOf[user]  += amount;
        totalSupply      += amount;
        emit Mint(user, amount, _nonce);
        emit Transfer(address(0), user, amount);
    }

    /// Burn appele par l'utilisateur pour initier le retour vers la source
    function burn(uint256 amount) external {
        require(balanceOf[msg.sender] >= amount, "Insufficient balance");
        balanceOf[msg.sender] -= amount;
        totalSupply           -= amount;
        emit Burn(msg.sender, amount);
        emit Transfer(msg.sender, address(0), amount);
    }
}
"#;

println!("=== TokenBridge (chaine source) ===");
println!("{}", TOKEN_BRIDGE);
println!("=== WrappedToken (chaine destination) ===");
println!("{}", WRAPPED_TOKEN);

=== TokenBridge (chaine source) ===


// SPDX-License-Identifier: MIT


pragma solidity ^0.8.28;


import "@openzeppelin/contracts/utils/cryptography/ECDSA.sol";


import "@openzeppelin/contracts/utils/cryptography/MessageHashUtils.sol";


/// @title TokenBridge - Chaine source (Lock & Unlock)


... (86 outputs left out)

        emit Burn(msg.sender, amount);


        emit Transfer(msg.sender, address(0), amount);


    }


}


### Interpretation : Securite Cross-Chain

Les systèmes cross-chain introduisent des vulnerabilites spécifiques qui n'existent pas dans les contrats single-chain.

| Vulnerabilite | Description | Impact | Mitigation |
|----------------|-------------|--------|------------|
| **Reentrancy cross-chain** | Une attaque sur une chain affecte l'autre | Drain de fonds | Verifier la finalite avant exécution |
| **Double-spend** | Même tokens utilises sur plusieurs chains | Inflation | Lock/Burn atomique avec preuves cryptographiques |
| **Oracle manipulation** | Fausse information sur l'etat d'une chain | Decisions basees sur données fausses | Multiples oracles, verification croisee |
| **Bridge exploits** | Bugs dans les contrats de bridge | Pertes massives (centaines de millions) | Audits, formals verification, limits |
| **Governance attacks** | Manipulation des votes cross-chain | Prise de contrôle illegitime | Snapshot voting, time locks |

**Points cles** :
1. La **finalite** est critique : attendre suffisamment de confirmations avant d'executer une action cross-chain
2. Les **rate limits** (limites quotidiennes) reduisent l'impact d'une exploitation (ex: max 1M\$ par jour)
3. L'**emergency pause** permet de geler un bridge si une attaque est detectee
4. Le **monitoring** en temps reel est essentiel pour detecter les activites suspectes

**Exemples d'exploits reels** :
- **Ronin Bridge (2022)** : 625M\$ voles - attaque sur les validators
- **Wormhole (2022)** : 320M\$ voles - bug de signature forgeable
- **Harmony Bridge (2022)** : 100M\$ voles - compromission des cles privees

> **Note technique** : La règle d'or des bridges : "Trust, but verify". Faire confiance au bridge ne suffit pas, il faut verifier la finalite, la signature, et l'etat de la chaîne source.

***

[<< Solana & Anchor](../05-Alternative-Chains/SC-22-Solana-Anchor-Python.ipynb) | [Testnet Deploy >>](SC-24-Testnet-Deploy-Python.ipynb) | [jumeau Python](SC-23-Cross-Chain-Python.ipynb)

**Indice : Exercice Bridge Token Cross-Chain**

Pour implementer un bridge securise, rappelez-vous les proprietes critiques d'un système cross-chain :

**Pattern Lock & Mint** :
1. **Source chain** : L'utilisateur appelle `lock(amount)` → les tokens sont transfères dans le contrat bridge (verrouilles)
2. **Detection** : Un oracle/relayer detecte l'événement `Lock` et prouve la transaction sur la destination
3. **Destination chain** : Le bridge appelle `mint(user, amount)` sur le `WrappedToken` → creation de tokens representatifs

**Securite critique - Protection contre replay attacks** :
- Le `nonce` doit etre unique et incrementer a chaque opération (timestamp ou compteur)
- La signature doit inclure : user, amount, nonce, chainId (pour eviter la reutilisation sur une autre chaîne)
- `validateSignature` doit verifier que le signataire est le validator autorise (pas d'utilisateur arbitraire)

**Indice implementation** :
- Utiliser `ECDSA.recover(hash, signature)` pour verifier le signataire
- Le hash a signer doit etre : `keccak256(abi.encodePacked(user, amount, nonce, chainId))`
- Stocker les nonces utilises dans un `mapping(uint256 => bool)` pour empecher la reutilisation
- `unlock` ne doit etre appelable que par le validator (pas par l'utilisateur directement)

**Cas d'erreur a gerer** :
- Le bridge destination est en pause (utilise `whenNotPaused`)
- La signature est invalide (revert)
- Le nonce a déjà ete utilise (revert)
- Le montant depasse la limite quotidienne (rate limiting)

## Resume et perspectives

Ce notebook a explore les defis et les solutions de l'interoperabilite entre blockchains. Nous avons etudie les problemes fondamentaux du cross-chain (finalite, securite, latence, cout, oracle), implemente des contrats Chainlink CCIP pour l'envoi et la reception de messages cross-chain avec transfert de tokens, examine les chain selectors et l'architecture de routage de Chainlink, et analyse les vulnerabilites spécifiques aux systèmes cross-chain (reentrancy cross-chain, double-spend, manipulation d'oracle, exploits de bridge, attaques de gouvernance).

L'enjeu central de l'interoperabilite est le compromis entre securite et performance : les protocols bases sur des oracles (CCIP) offrent une securite elevee au prix d'une latence plus importante, tandis que les solutions ultra-legers (LayerZero) privilegient la rapidite avec un modèle de confiance différent. Les exploits historiques majeurs (Ronin Bridge 625M\$, Wormhole 320M\$, Harmony 100M\$) rappellent que les bridges concentrent un risque systemique considerable et qu'une verification rigoureuse de la finalite, des rate limits et des mécanismes de pause d'urgence sont indispensables.

Le notebook suivant, [SC-24-Testnet-Deploy-Python](SC-24-Testnet-Deploy-Python.ipynb), aborde le deploiement concret de contrats sur des reseaux de test publics (Sepolia, XRP Testnet) avant le passage en production.

> Carnet issu de l'EPIC #26534.